In [3]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from utiles import read_frames, preprocess

df = pd.read_csv("index.csv")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(df), device)

: 

In [7]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()   # выход: 512 признаков на кадр
model.eval().to(device)    # eval: BatchNorm в режиме инференса

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/ekaterina/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [8]:
feats = []
with torch.no_grad():                      # без градиентов: быстрее и меньше памяти
    for path in df.path:
        x = preprocess(read_frames(path)).to(device)   # (16, 3, 224, 224)
        feats.append(model(x).cpu().numpy())           # (16, 512)

feats = np.stack(feats)                    # (272, 16, 512)
np.save("features.npy", feats)
print(feats.shape)

(272, 16, 512)


In [1]:
from pathlib import Path
import cv2

def read_all(path):
    files = sorted(Path(path).glob("*.jpg"), key=lambda f: int(f.stem))
    return np.stack([cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB) for f in files])

def clip_windows(frames, win=16, stride=8):
    T = len(frames)
    if T < win:                                   # короткий клип: растягиваем до 16
        frames = frames[np.linspace(0, T - 1, win).round().astype(int)]
        T = win
    starts = range(0, T - win + 1, stride)
    return torch.stack([tf(torch.from_numpy(frames[s:s + win]).permute(0, 3, 1, 2))
                        for s in starts])        # (окна, 3, 16, 112, 112)

feats = []
with torch.no_grad():
    for path in df.path:
        w = clip_windows(read_all(path))
        out = torch.cat([model(b.to(device)).cpu() for b in w.split(8)])  # по 8 окон за раз (4 ГБ)
        feats.append(out.mean(0).numpy())        # среднее по окнам -> (512,)

feats = np.stack(feats)
np.save("features_r21d_win.npy", feats)
print(feats.shape)   # (272, 512)

NameError: name 'torch' is not defined